In [ ]:
# Cell 1: Clone or pull latest code from GitHub
import os
REPO = "https://github.com/Elijahzyp/loomguard.git"
REPO_DIR = "/content/loomguard"
if not os.path.exists(REPO_DIR):
    !git clone {REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}/experiments/phase1a_aitex
print("✅ Repo ready")

In [ ]:
# Cell 2: Mount Google Drive for data input and results output
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs("/content/drive/MyDrive/loomguard_data/results", exist_ok=True)
print("✅ Drive mounted")
print("Data:", os.path.exists("/content/drive/MyDrive/loomguard_data/prepared"))

In [ ]:
# Cell 3: Install dependencies
!pip install timm anomalib scikit-learn tqdm -q
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.is_available()}")

In [ ]:
# Cell 4: Run ResNet18 training — COLAB_MODE auto-detected, no manual CONFIG change needed
%run scripts/train_resnet.py

In [ ]:
# Cell 5: Run PatchCore training — COLAB_MODE auto-detected, no manual CONFIG change needed
%run scripts/train_patchcore.py

In [ ]:
# Cell 6: Frozen baseline complete
# (runtime.unassign removed — sweep cells follow below)
print("✅ Phase 1A frozen baseline section ends here")
print("Cells 7+ are Phase 2 scaffolding and sweep")


In [ ]:
# Cell 7: install new pipeline deps (additive — does not affect cells 1–6)
!pip install albumentations pyyaml pandas -q
print('✅ new deps ready')

In [ ]:
# Cell 8: example unified train run (Exp 1 — conservative augmentation)
%cd /content/loomguard/experiments/phase1a_aitex
!python -m scripts.classification.train \
  --config scripts/config/exp01_aug.yaml \
  --seed 42 \
  --tag exp01_aug_conservative

In [ ]:
# Cell 9: anomaly-detection scaffold check — EfficientAD (do not tune on AITEX)
!python -m scripts.anomaly.anomaly_efficientad

In [ ]:
# Cell 10: aggregate per-experiment summary CSVs from results/runs/
!python scripts/aggregate_results.py
import os; print('summary dir:', os.listdir('results/summary') if os.path.exists('results/summary') else '(none)')

In [ ]:
# Cell 11: Sweep config + helpers (no GPU work).
# Defines the experiment matrix, paths, and idempotency/sync helpers used by
# Cells 12-16. Re-run safe.
import os, subprocess, time
from pathlib import Path

# --- experiment matrix ---
SEEDS = [42, 123, 2024]
AUG_PROFILES = ["none", "conservative"]
TTA_MODES = ["4way", "8way"]
EPOCHS = 30  # match frozen baseline depth

# --- paths ---
REPO_DIR = Path("/content/loomguard/experiments/phase1a_aitex")
RUNS_DIR = REPO_DIR / "results" / "runs"
SUMMARY_DIR = REPO_DIR / "results" / "summary"
DRIVE_BASE = Path("/content/drive/MyDrive/loomguard_data/results")
DRIVE_RUNS = DRIVE_BASE / "runs"
DRIVE_SUMMARY = DRIVE_BASE / "summary"

os.chdir(REPO_DIR)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

# --- idempotency / sync helpers ---
def _existing_run(tag, seed):
    """Return run_dir Path if metrics.json exists for this tag+seed (Drive or local)."""
    # 优先查 Drive（train.py COLAB_MODE 写到这里）
    for d in sorted(DRIVE_RUNS.glob(f"*_{tag}_seed{seed}")):
        if (d / "metrics.json").exists():
            return d
    # fallback 查本地
    for d in sorted(RUNS_DIR.glob(f"*_{tag}_seed{seed}")):
        if (d / "metrics.json").exists():
            return d
    return None

def _rsync_run(run_dir):
    subprocess.run(
        ["rsync", "-a", f"{run_dir}/", f"{DRIVE_RUNS / run_dir.name}/"],
        check=False,
    )

def _rsync_summary():
    DRIVE_SUMMARY.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["rsync", "-a", f"{SUMMARY_DIR}/", f"{DRIVE_SUMMARY}/"],
        check=False,
    )

# --- print plan ---
N_TRAIN = len(SEEDS) * len(AUG_PROFILES)
N_TTA = N_TRAIN * len(TTA_MODES)
print("Sweep plan:")
print(f"  seeds:        {SEEDS}")
print(f"  aug profiles: {AUG_PROFILES}")
print(f"  TTA modes:    {TTA_MODES}")
print(f"  epochs/run:   {EPOCHS}")
print(f"  -> {N_TRAIN} trainings + {N_TTA} TTA evals = {N_TRAIN + N_TTA} jobs")


In [ ]:
# Cell 12: Training sweep — aug × seed = 6 runs.
# Outer loop = aug, inner loop = seed (so all 3 seeds of aug=none finish before
# aug=conservative starts; partial sweeps still yield a complete control group).
# - Idempotent: skips if (tag, seed) already has a metrics.json
# - Per-run rsync to Drive
# - Per-run try/except so a single failure doesn't kill the sweep
succeeded, failed = [], []
job = 0
for aug in AUG_PROFILES:
    for seed in SEEDS:
        job += 1
        tag = f"exp01_aug_{aug}"
        existing = _existing_run(tag, seed)
        if existing is not None:
            print(f"[{job}/{N_TRAIN}] aug={aug} seed={seed} SKIP (found {existing.name})")
            succeeded.append(existing)
            continue
        print(f"[{job}/{N_TRAIN}] aug={aug} seed={seed} starting...")
        t0 = time.time()
        cmd = [
            "python", "-m", "scripts.classification.train",
            "--config", "scripts/config/default.yaml",
            "--tag", tag,
            "--seed", str(seed),
            "--aug-profile", aug,
            "--epochs", str(EPOCHS),
        ]
        try:
            subprocess.run(cmd, check=True)
            run_dir = _existing_run(tag, seed)
            if run_dir is None:
                raise RuntimeError("training exited 0 but no metrics.json found")
            if not str(run_dir).startswith(str(DRIVE_RUNS)):
                _rsync_run(run_dir)
            succeeded.append(run_dir)
            print(f"  done in {time.time() - t0:.0f}s -> {run_dir.name}")
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            failed.append((tag, seed, str(e)))

print(f"\nTraining sweep done. {len(succeeded)}/{N_TRAIN} succeeded.")
if failed:
    print(f"Failures: {failed}")


In [ ]:
# Cell 13: Sanity check — confirm Cell 12 produced N_TRAIN run dirs with metrics.json
matches = sorted(RUNS_DIR.glob("*_exp01_aug_*_seed*"))
print(f"Found {len(matches)} exp01_aug_* run dirs (expected {N_TRAIN}):")
for d in matches:
    has_metrics = (d / "metrics.json").exists()
    print(f"  {d.name}  metrics.json={'OK' if has_metrics else 'MISSING'}")


In [ ]:
# Cell 14: TTA sweep — for each completed exp01_aug_* run, run 4way + 8way TTA.
# Filtered glob (exp01_aug_* prefix) so historical runs and baseline_repro are
# not pulled in. Idempotent on metrics_tta_<mode>.json. No --force needed: any
# pre-existing TTA file means a previous sweep ran here, so skip.
sweep_runs = sorted(DRIVE_RUNS.glob("*_exp01_aug_*_seed*"))
sweep_runs = [d for d in sweep_runs if (d / "metrics.json").exists()]
if not sweep_runs:
    sweep_runs = sorted(RUNS_DIR.glob("*_exp01_aug_*_seed*"))
    sweep_runs = [d for d in sweep_runs if (d / "metrics.json").exists()]
total = len(sweep_runs) * len(TTA_MODES)
print(f"TTA sweep over {len(sweep_runs)} run dirs × {len(TTA_MODES)} modes = {total} jobs")

tta_succeeded, tta_failed = [], []
job = 0
for run_dir in sweep_runs:
    for mode in TTA_MODES:
        job += 1
        out = run_dir / f"metrics_tta_{mode}.json"
        if out.exists():
            print(f"[{job}/{total}] {run_dir.name} mode={mode} SKIP (exists)")
            tta_succeeded.append((run_dir, mode))
            continue
        print(f"[{job}/{total}] {run_dir.name} mode={mode} starting...")
        t0 = time.time()
        cmd = [
            "python", "-m", "scripts.classification.tta_eval",
            "--run-dir", str(run_dir),
            "--tta-mode", mode,
        ]
        try:
            subprocess.run(cmd, check=True)
            if not str(run_dir).startswith(str(DRIVE_RUNS)):
                _rsync_run(run_dir)
            tta_succeeded.append((run_dir, mode))
            print(f"  done in {time.time() - t0:.0f}s")
        except Exception as e:
            print(f"  FAILED: {type(e).__name__}: {e}")
            tta_failed.append((run_dir.name, mode, str(e)))

print(f"\nTTA sweep done. {len(tta_succeeded)}/{total} succeeded.")
if tta_failed:
    print(f"Failures: {tta_failed}")


In [ ]:
# Symlink DRIVE_RUNS exp01 dirs into RUNS_DIR so aggregate_results.py can find them
for d in DRIVE_RUNS.glob("*_exp01_aug_*_seed*"):
    link = RUNS_DIR / d.name
    if not link.exists():
        link.symlink_to(d)

# Cell 15: Aggregate summary CSVs and display key tables (no GPU)
import pandas as pd

subprocess.run(["python", "scripts/aggregate_results.py"], check=False)
_rsync_summary()

for name in ["exp01_aug_comparison.csv", "exp02_tta_comparison.csv"]:
    p = SUMMARY_DIR / name
    if not p.exists():
        print(f"(missing) {name}")
        continue
    print(f"\n=== {name} ===")
    df = pd.read_csv(p)
    cols = [c for c in [
        "tag", "seed", "auroc", "augmentation_profile", "tta_mode",
        "delta_auroc", "best_val_auroc_during_training",
    ] if c in df.columns]
    print(df[cols].to_string(index=False))

# Hand-rolled pivot: aug × seed AUROC matrix (training run, no TTA)
exp01_csv = SUMMARY_DIR / "exp01_aug_comparison.csv"
if exp01_csv.exists():
    df = pd.read_csv(exp01_csv)
    needed = {"augmentation_profile", "seed", "auroc"}
    if needed.issubset(df.columns):
        # Filter out TTA rows that may have been routed here by tag prefix accident
        if "tta_mode" in df.columns:
            df = df[df["tta_mode"].fillna("none").isin(["", "none"])]
        pivot = df.pivot_table(index="augmentation_profile", columns="seed", values="auroc")
        print("\n=== AUROC pivot (aug × seed) — training run, no TTA ===")
        print(pivot.round(4).to_string())


In [ ]:
# Cell 16: Final sync to Drive and disconnect.
# Belt-and-suspenders rsync in case any prior cell missed a sync.
for run_dir in sorted(RUNS_DIR.glob("*_exp01_aug_*_seed*")):
    _rsync_run(run_dir)
_rsync_summary()
n_runs = len([d for d in RUNS_DIR.iterdir() if d.is_dir()])
print(f"Sweep complete. Total run dirs in results/runs/: {n_runs}. Drive synced.")
print("Disconnecting in 10s...")
time.sleep(10)
from google.colab import runtime
runtime.unassign()
